In [ ]:
import pandas as pd
import unicodedata
from collections import Counter
import re

In [ ]:
df = pd.read_csv('train.csv',encoding = 'utf-8-sig')
df = df.dropna()

In [ ]:
def decompose_char(ch):
    return unicodedata.normalize("NFD",ch)

def decompose_text(text):
    return [decompose_char(ch) for ch in text]

In [ ]:
CHOSUNG_START = 0x1100
JUNGSUNG_START = 0x1161
JONGSUNG_START = 0x11A8

In [ ]:
def get_jamo_type(c):
    """문자가 초성/중성/종성인지 판별."""
    code = ord(c)
    if 0x1100 <= code <= 0x115F:
        return "초성"
    elif 0x1160 <= code <= 0x11A7:
        return "중성"
    elif 0x11A8 <= code <= 0x11FF:
        return "종성"
    else:
        return "기타"

In [ ]:
def get_differences(inp_dec, out_dec):
    diffs = []
    min_len = min(len(inp_dec), len(out_dec))

    for i in range(min_len):
        if inp_dec[i] != out_dec[i]:
            diffs.append((inp_dec[i], out_dec[i]))

    # 길이가 다르면 나머지도 표시
    if len(inp_dec) != len(out_dec):
        diffs.append(("len_diff", f"{len(inp_dec)} vs {len(out_dec)}"))

    return diffs

In [ ]:
comparison_rows = []


for i in range(30):  # 처음 30개만 예시
    inp = df.iloc[i]["input"]
    out = df.iloc[i]["output"]

    inp_dec = decompose_text(inp)
    out_dec = decompose_text(out)

    diffs = get_differences(inp_dec, out_dec)

    comparison_rows.append({
        "input": inp,
        "output": out,
        "input_decomposed": inp_dec,
        "output_decomposed": out_dec,
        "differences": diffs
    })

In [ ]:
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.head()

,input,output,input_decomposed,output_decomposed,differences
0,별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈...,별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자...,"[별, , 한, , 게, 토, , 았, 깝, 땀, ., ...","[별, , 한, , 개, 도, , 아, 깝, 다, ., ,...","[(게, 개), (토, 도), (았, 아), (땀, 다), (싸..."
1,잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ,잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ,"[잚, 많, , 쟉, 꼬, , 갉, , 태, , 좋, ...","[잠, 만, , 자, 고, , 갈, , 때, , 좋, ᄂ...","[(잚, 잠), (많, 만), (쟉, 자), (꼬, 고), ..."
2,절테 간면 않 된는 굣 멥몫,절대 가면 안 되는 곳 메모,"[절, 테, , 간, 면, , 않, , 된, 는, ,...","[절, 대, , 가, 면, , 안, , 되, 는, , ᄀ...","[(테, 대), (간, 가), (않, 안), (된, 되), (..."
3,야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵...,아... 가격 좋고 뷰도 뻥 뚫려서 시원하지만 담배 냄새 미쳐버림. 싸게 하루만 묵...,"[야, ., ., ., , 칵, 컥, , 좋, 꾜, , 부, ...","[아, ., ., ., , 가, 격, , 좋, 고, , 뷰, ᄃ...","[(야, 아), (칵, 가), (컥, 격), (꾜, 고), (ᄇ..."
4,집윈 축쳐눌료 딴너왓눈뎁 카셩뷔 좋곱 칼쿰한네올. 쩌럼한뒈 뮬콰 욺료토 잊쿄 빻토 ...,지인 추천으로 다녀왔는데 가성비 좋고 깔끔하네요. 저렴한데 물과 음료도 있고 방도 ...,"[집, 윈, , 축, 쳐, 눌, 료, , 딴, 너, 와...","[지, 인, , 추, 천, 으, 로, , 다, 녀, 왔, ...","[(집, 지), (윈, 인), (축, 추), (쳐, 천), ..."


In [ ]:
all_diffs = []

for inp, out in zip(df["input"], df["output"]):
    inp_dec = [decompose_char(c) for c in inp]
    out_dec = [decompose_char(c) for c in out]

    min_len = min(len(inp_dec), len(out_dec))

    for i in range(min_len):
        if inp_dec[i] != out_dec[i]:
            all_diffs.append((inp_dec[i], out_dec[i]))

    # 길이 차이도 기록
    if len(inp_dec) != len(out_dec):
        all_diffs.append(("len_diff", f"{len(inp_dec)} vs {len(out_dec)}"))

# 등장 횟수 카운트
diff_counter = Counter(all_diffs)

# 상위 50개 보기
for pair, cnt in diff_counter.most_common(50):
    print(pair, cnt)

('눈', '는') 8244
('오', '요') 5655
('뉘', '니') 5572
('댜', '다') 5107
('교', '고') 5031
('위', '이') 5011
('됴', '도') 3753
('햐', '하') 3732
('학', '하') 3332
('많', '만') 3208
('따', '다') 3083
('써', '서') 3079
('타', '다') 3076
('꼬', '고') 3045
('셔', '서') 3012
('코', '고') 3007
('갸', '가') 2858
('료', '로') 2671
('운', '은') 2507
('토', '도') 2356
('먼', '면') 2300
('또', '도') 2209
('뤼', '리') 2148
('씁', '습') 2083
('예', '에') 2042
('윈', '인') 2024
('뮤', '무') 2018
('웨', '에') 1910
('쟝', '장') 1835
('까', '가') 1835
('카', '가') 1831
('여', '어') 1807
('움', '음') 1781
('쥐', '지') 1697
('헤', '해') 1681
('얀', '안') 1679
('야', '아') 1651
('잊', '있') 1633
('한', '하') 1529
('랴', '라') 1471
('숩', '습') 1469
('뗄', '텔') 1409
('효', '호') 1401
('냐', '나') 1398
('옹', '용') 1380
('잇', '있') 1344
('핫', '하') 1307
('뮨', '문') 1287
('졍', '정') 1282
('귀', '기') 1279


In [ ]:
# 등장 횟수가 특정 threshold 이상인 변화만 규칙으로 사용
threshold = 100

rule_dict = {}

for (inp_jamo, out_jamo), cnt in diff_counter.items():
    if inp_jamo != "len_diff" and cnt >= threshold:
        rule_dict[inp_jamo] = out_jamo

len(rule_dict), list(rule_dict.items())[:20]


(1004,
 [('게', '개'),
  ('토', '또'),
  ('았', '아'),
  ('땀', '다'),
  ('싸', '사'),
  ('펼', '별'),
  ('캐', '깨'),
  ('눈', '은'),
  ('징', '지'),
  ('롯', '로'),
  ('섞', '서'),
  ('맒', '말'),
  ('록', '로'),
  ('섧', '서'),
  ('멍', '명'),
  ('핥', '할'),
  ('쟈', '자'),
  ('닐', '니'),
  ('룐', '론'),
  ('녀', '너')])

In [ ]:
def apply_rules(text, rule_dict):
    result = []

    for ch in text:
        dec = unicodedata.normalize("NFD", ch)

        # 규칙이 있으면 변환
        if dec in rule_dict:
            dec = rule_dict[dec]

        # 다시 한글 조합
        rec = unicodedata.normalize("NFC", dec)
        result.append(rec)

    return "".join(result)

In [ ]:
correct = 0
total = len(df)

for inp, out in zip(df["input"], df["output"]):
    converted = apply_rules(inp, rule_dict)
    if converted == out:
        correct += 1

accuracy = correct / total
accuracy

0.0005327177483796502

In [ ]:
df['input_cleaned'] = df['input'].apply(lambda x: apply_rules(x, rule_dict))
df[['input', 'input_cleaned', 'output']].head(20)


,input,input_cleaned,output
0,별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈...,별 하 개또 아가다. 외 사라듯리 별 1깨을 준은지 격꺾폰 사라믐로서 말로 서명할자...,별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자...
1,잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ,잘만 작고 가 대 좋에요. 자못도 줆 ㅋ,잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ
2,절테 간면 않 된는 굣 멥몫,절태 가먼 아 되은 고 멥모,절대 가면 안 되는 곳 메모
3,야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵...,았... 각격 좋교 뷰도 평 뚫렷서 시원아지많 다패 네센 밌쪄벽리. 샥게 할루많 무...,아... 가격 좋고 뷰도 뻥 뚫려서 시원하지만 담배 냄새 미쳐버림. 싸게 하루만 묵...
4,집윈 축쳐눌료 딴너왓눈뎁 카셩뷔 좋곱 칼쿰한네올. 쩌럼한뒈 뮬콰 욺료토 잊쿄 빻토 ...,집위 축처을로 단너와은대 까성비 좋고 깔끔하내요. 처렴하되 무과 을로또 이교 방또 ...,지인 추천으로 다녀왔는데 가성비 좋고 깔끔하네요. 저렴한데 물과 음료도 있고 방도 ...
5,펀냔휜 잘 쉭곶 왔쑵닝따. 준윙에 맏쥡됴 만학썩 좋흖 겼 갇따용.,번나휜 잘 시고 와습니타. 준이에 마집도 많하서 좋은 것 가타요.,편안히 잘 쉬고 왔습니다. 주위에 맛집도 많아서 좋은 것 같아요.
6,"쓰윔튿룸위 깝써칙갖 있네 ㅋ 꼭굽 췸규웨 낑펙투, 합뻔퓨 옥쪽, 캬뻬 깥은 띰 떼잎...","스이튿름있 가어지같 이내 ㅋ 고급 지구외 낑펙트, 하번뷰 오족, 까페 가은 띰 대입...","스위트룸의 값어치가 있네 ㅋ 고급 침구에 킹베드, 하버뷰 욕조, 카페 같은 티 테이..."
7,쨉빵뮨 의싸 쩐허 없쓺. 씬랏슬톄잎뾰댜 떠 짝음. 홈뗄 줄찾짱 15땜만 캉눙. 혹텔...,쨉방무 의사 천허 어음. 시라슬데입보다 터 작음. 호텔 줄차창 15땜많 캉능. 호텔...,재방문 의사 전혀 없음. 신라스테이보다 더 작음. 호텔 주차장 15대만 가능. 호텔...
8,념묵 멎쥑교 꽁귀 좋습니닸. 췐곡윕뉘댜!,넘무 멎직고 고이 좋습니다. 췐고위이다!,너무 멋지고 공기 좋습니다. 최고입니다!
9,"짱졈: 쩡켤함, 츄챠 씨셜 죠흠, 쉼섦 좋음. 단젊: 줏짢 츌짜할 떼 뷸뻔함, ...","창점: 정결하, 추차 이서 조음, 심서 좋음. 다절: 주찮 출자하 대 불번하, ...","장점: 청결함, 주차 시설 좋음, 시설 좋음. 단점: 주차 출차할 때 불편함, ..."


In [ ]:
for i in range(20):
    print("input:        ", df.iloc[i]["input"])
    print("input_cleaned:", df.iloc[i]["input_cleaned"])
    print("output:       ", df.iloc[i]["output"])
    print("-" * 50)

input:         별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈닐 탯끎룐눈 녀뮤 퀼교... 야뭍툰 둠 변 닺씨 깍낄 싫훈 굣. 깸삥읊 20여 년 댜녁뵨 곧 중 쩨윌 귑푼 낙팠떤 곶.
input_cleaned: 별 하 개또 아가다. 외 사라듯리 별 1깨을 준은지 격꺾폰 사라믐로서 말로 서명할자니 탯끔론은 너무 퀼고... 았무툰 둠 번 다이 각낄 실은 고. 캠핑으 20어 년 다녁보 고 중 체위 기분 나팠던 고.
output:        별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자니 댓글로는 너무 길고... 아무튼 두 번 다시 가길 싫은 곳. 캠핑을 20여 년 다녀본 곳 중 제일 기분 나빴던 곳.
--------------------------------------------------
input:         잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ
input_cleaned: 잘만 작고 가 대 좋에요. 자못도 줆 ㅋ
output:        잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ
--------------------------------------------------
input:         절테 간면 않 된는 굣 멥몫
input_cleaned: 절태 가먼 아 되은 고 멥모
output:        절대 가면 안 되는 곳 메모
--------------------------------------------------
input:         야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵겠댜! 한눈 쌀람한뗌많 쭈쳔. 탐패 냄쌕갊 묘둔 쟝졈울 까저갼눈 콧. 놂랙팡엣셔 칵좋 닮패왕 윳흥예 천렸욹 택 냐눈 넴쌘갸 꼐쏙 방웨 잊슴 ㅆ... 샨닉깎 할 맑 엽숨.
input_cleaned: 았... 각격 좋교 뷰도 평 뚫렷서 시원아지많 다패 네센 밌쪄벽리. 샥게 할루많 무겠다! 하은 사라하때만 주전. 다패 냄쌕갈 모든 장점으 카져간은 고. 

In [ ]:
df["input"] = df["input_cleaned"]
df = df.drop(columns=["input_cleaned"])

In [ ]:
df.to_csv("train_cleaned.csv", index=False, encoding="utf-8-sig")
